#<h1 align="center">**LMF Interactive Scatterplot**</h1>




<div align="justify">

This is an interactive scatter plot where you can easily explore all the processed samples from the Low Methane Forages project.

First go to **File** → **"Save a copy in Drive"**. This will create a copy of the notebook in your Google Drive. You can then edit the notebook and explore the data using the interactive controls.

To activate the visualization options, click the run_button.jpg button located at the top of the panel. Then, explore different combinations of X and Y axes, subsets, and functional groups. Once you have adjusted the plot to your preference, you can export it by clicking the three_dots_button.jpg button in the upper-right corner of the scatter plot.

**Advanced options**: You can customize the benchmark visualization by adding one or more id_lab values to the list in the second cell of the **Plot code** section. Benchmark samples will be highlighted in **purple**.

To highlight one or more samples of interest, add their id_lab values to the target list. These samples will be displayed in **red**.

Don't know the id_lab of your sample of interest? Explore the complete dataset [here.](https://github.com/maurope/lmf/blob/main/output/2026_05_22_data_delivery/08_dashboard_june_2026/compiled_dashboard_2026_7_10.csv)

</div>


# 1.0 Import libraries

In [1]:
import pandas as pd
import altair as alt
import ipywidgets as widgets
from IPython.display import display

#2.0  Data load

In [2]:
url = "https://raw.githubusercontent.com/maurope/lmf/main/data/2026_08_20_database_categories_quartiles__visualization_public/compiled_categories_quartiles.csv"
df = pd.read_csv(url)
df

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,n_replicates_nutrition,dm_percentage,ash_dm,...,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm,ch4_category,tddm_category,lmf_category,lmf_category_rank,quartile,quartile_rank
0,F24-3470,CIAT-11194,1,199.0,Genetic_bank,Stylosanthes hamata,Herbaceous_legumes,2,92.89,9.61,...,15.17,18.02,46.12,59.25,Medium,Not High,Category_2,81.0,Q1,58.0
1,F24-3471,CIAT-11999,1,201.0,Genetic_bank,Stylosanthes guianensis,Herbaceous_legumes,2,93.50,10.46,...,12.98,16.25,46.61,58.55,Medium,Not High,Category_2,88.0,Q1,66.0
2,F24-3472,CIAT-12318,1,203.0,Genetic_bank,Stylosanthes hamata,Herbaceous_legumes,2,94.10,11.08,...,13.99,16.96,52.27,56.88,Medium,Not High,Category_2,148.0,Q2,78.0
3,F24-3427,CIAT-1257,1,113.0,Genetic_bank,Stylosanthes scabra,Herbaceous_legumes,2,92.72,9.52,...,15.71,17.60,46.32,58.95,Medium,Not High,Category_2,84.0,Q1,62.0
4,F24-3473,CIAT-13575,1,205.0,Genetic_bank,Desmodium incanum,Herbaceous_legumes,2,94.14,11.52,...,13.60,16.23,42.43,42.85,Low,Not High,Category_2,46.0,Q3,224.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
688,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Medium,Not High,Category_2,151.0,NaN,NaN
689,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Low,Not High,Category_2,1.0,NaN,NaN
690,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,High,Not High,Category_4,36.0,NaN,NaN
691,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Low,Not High,Category_2,55.0,NaN,NaN


# 3.0 Plot code

In [3]:
df = df.copy()
df["subset"] = df["subset"].astype(str)
df["functional_group"] = df["functional_group"].fillna("Unknown")

In [14]:
# ---------------------------------------------
# Lists of special samples
# ---------------------------------------------

target = ["CIAT-PM-21-2580"]
star_grass_control_ids = ["Sample-8"]
benchmark_ids = ["CIAT-6962-Mombaza-Ex","CIAT-606-Basilisk-Opt","CIAT-6294-Marandu-Opt","CIAT-36087-MulatoII-Opt",
                 "CIAT-6133-Llanero-Opt","CIAT-6962-Mombaza-Opt","Paja-Sabana-Madura","Humidicola-679","Sabana-Quemada",
                 "CIAT-6294-Marandu-Exc", "CIAT-606-Basilisk-Exc", "BR-02-1752-Cayman-Exc", "BR-06-423-Cayman-Exc"]

In [37]:
# ============================================================
# Widgets
# ============================================================

numeric_columns = sorted(df.select_dtypes(include="number").columns.tolist())

# default values
default_x = "tddm"
default_y = "methane_intensity"

# if any column does not exist, use the first
if default_x not in numeric_columns:
    default_x = numeric_columns[0]

if default_y not in numeric_columns:
    default_y = numeric_columns[1] if len(numeric_columns) > 1 else numeric_columns[0]

x_widget = widgets.Dropdown(
    options=numeric_columns,
    value=default_x,
    description="X:"
)

y_widget = widgets.Dropdown(
    options=numeric_columns,
    value=default_y,
    description="Y:"
)

subset_widget = widgets.Dropdown(
    options=["All"] + sorted(df["subset"].astype(str).unique().tolist()),
    value="All",
    description="Subset:"
)

functional_widget = widgets.Dropdown(
    options=["All"] + sorted(df["functional_group"].dropna().unique().tolist()),
    value="All",
    description="Group:"
)

category_widget = widgets.Dropdown(
    options=["All"] + sorted(df["lmf_category"].dropna().unique().tolist()),
    value="All",
    description="Category:"
)

quartile_widget = widgets.Dropdown(
    options=["All"] + sorted(df["quartile"].dropna().unique().tolist()),
    value="All",
    description="Quartile:"
)

# ============================================================
# Plot function
# ============================================================

def plot(x, y, subset, functional_group, Category, Quartile):

    data = df.copy()

    # Filter by subset
    if subset != "All":
        data = data[data["subset"].astype(str) == subset]

    # Filter by functional group
    if functional_group != "All":
        data = data[data["functional_group"] == functional_group]

    # Filter by category
    if Category != "All":
        data = data[data["lmf_category"] == Category]

    # Filter by quartile
    if Quartile != "All":
        data = data[data["quartile"] == Quartile]

    # Create plotting groups
    data["plot_group"] = data["functional_group"]

    data.loc[
        data["id"].isin(benchmark_ids),
        "plot_group"
    ] = "Benchmark"

    data.loc[
        data["id"].isin(star_grass_control_ids),
        "plot_group"
    ] = "Star Grass Control"

    data.loc[
        data["id"].isin(target),
        "plot_group"
    ] = "Target"

    # Scatter plot
    chart = (
        alt.Chart(data)
        .mark_circle(size=70)
        .encode(
            x=alt.X(
                x,
                title=x.replace("_", " ").title()
            ),
            y=alt.Y(
                y,
                title=y.replace("_", " ").title()
            ),
            color=alt.Color(
                "plot_group:N",
                scale=alt.Scale(
                    domain=[
                        "Herbaceous_legumes",
                        "Grasses",
                        "Shrub_Trees",
                        "Benchmark",
                        "Star Grass Control",
                        "Target",
                    ],
                    range=[
                        "green",
                        "cornflowerblue",
                        "orange",
                        "purple",
                        "black",
                        "red",
                    ],
                ),
                legend=alt.Legend(title="Sample Type"),
            ),
            tooltip=[
                "id",
                "id_lab",
                "tax_name",
                "functional_group",
                "subset",
                "n_replicates_nutrition",
                "dm_percentage",
                "ash_dm",
                "om_percentage",
                "pc_percentage_dm",
                "adf_percentage_dm",
                "ndf_percentage_dm",
                "n_replicates_gas",
                "ch4_percentage_in_gas_8h",
                "ch4_percentage_in_gas_24h",
                "methane_intensity",
                "tddm",
                "ch4_category",
                "tddm_category",
                "lmf_category",
                "lmf_category_rank",
                "quartile",
                "quartile_rank"

            ],
        )
        .properties(
            width=850,
            height=600,
            title=f"{y.replace('_',' ').title()} vs {x.replace('_',' ').title()}",
        )
        .configure_axis(
            labelFontSize=14,
            titleFontSize=18,
        )
        .configure_legend(
            titleFontSize=16,
            labelFontSize=14,
        )
        .configure_title(
            fontSize=20,
        )
        .interactive()
    )

    display(chart)

# ============================================================
# Interactive dashboard
# ============================================================

controls = widgets.VBox([
    widgets.HBox([x_widget, y_widget]),
    widgets.HBox([subset_widget, functional_widget]),
    widgets.HBox([category_widget, quartile_widget]),
])

output = widgets.interactive_output(
    plot,
    {
        "x": x_widget,
        "y": y_widget,
        "subset": subset_widget,
        "functional_group": functional_widget,
        "Category":category_widget,
        "Quartile":quartile_widget
    },
)


#4.0  Interactive dashboard

In [38]:
display(controls, output)

Output()

### DataFrame con datos filtrados

Este código creará un nuevo DataFrame (`filtered_df`) aplicando los filtros seleccionados actualmente en los widgets. Ejecuta esta celda para ver los datos filtrados.

In [36]:
filtered_df = df.copy()

# Aplicar filtros basados en los valores actuales de los widgets
if subset_widget.value != "All":
    filtered_df = filtered_df[filtered_df["subset"].astype(str) == subset_widget.value]

if functional_widget.value != "All":
    filtered_df = filtered_df[filtered_df["functional_group"] == functional_widget.value]

if category_widget.value != "All":
    filtered_df = filtered_df[filtered_df["lmf_category"] == category_widget.value]

if quartile_widget.value != "All":
    filtered_df = filtered_df[filtered_df["quartile"] == quartile_widget.value]

display(filtered_df)

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,n_replicates_nutrition,dm_percentage,ash_dm,...,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm,ch4_category,tddm_category,lmf_category,lmf_category_rank,quartile,quartile_rank
0,F24-3470,CIAT-11194,1,199.0,Genetic_bank,Stylosanthes hamata,Herbaceous_legumes,2,92.89,9.61,...,15.17,18.02,46.12,59.25,Medium,Not High,Category_2,81.0,Q1,58.0
1,F24-3471,CIAT-11999,1,201.0,Genetic_bank,Stylosanthes guianensis,Herbaceous_legumes,2,93.50,10.46,...,12.98,16.25,46.61,58.55,Medium,Not High,Category_2,88.0,Q1,66.0
2,F24-3472,CIAT-12318,1,203.0,Genetic_bank,Stylosanthes hamata,Herbaceous_legumes,2,94.10,11.08,...,13.99,16.96,52.27,56.88,Medium,Not High,Category_2,148.0,Q2,78.0
3,F24-3427,CIAT-1257,1,113.0,Genetic_bank,Stylosanthes scabra,Herbaceous_legumes,2,92.72,9.52,...,15.71,17.60,46.32,58.95,Medium,Not High,Category_2,84.0,Q1,62.0
4,F24-3473,CIAT-13575,1,205.0,Genetic_bank,Desmodium incanum,Herbaceous_legumes,2,94.14,11.52,...,13.60,16.23,42.43,42.85,Low,Not High,Category_2,46.0,Q3,224.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
688,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Medium,Not High,Category_2,151.0,NaN,NaN
689,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Low,Not High,Category_2,1.0,NaN,NaN
690,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,High,Not High,Category_4,36.0,NaN,NaN
691,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Low,Not High,Category_2,55.0,NaN,NaN
